**Install RDKit**

In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 66.4 MB/s eta 0:00:00


**Verify RDKit installation**

In [ ]:
from rdkit import Chem
print(Chem.rdBase.rdkitVersion)

2026.03.3


**Import required libraries**

In [ ]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors

**Upload the cleaned dataset**

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving master_clean_standardized_Notebook2.csv to master_clean_standardized_Notebook2.csv


**Load the cleaned dataset**

In [ ]:
import pandas as pd

master = pd.read_csv("master_clean_standardized_Notebook2.csv")

print(master.shape)
master.head()

(8622, 4)


,SMILES,Label,Source,Standard_SMILES
0,Cc1cc(O)c2C(=O)c3c(O)cc(O)cc3C(=O)c2c1O,1,OASIS,Cc1cc(O)c2c(c1O)C(=O)c1cc(O)cc(O)c1C2=O
1,CC1CCC(CC=1)C(C)(C)O,0,OASIS,CC1=CCC(C(C)(C)O)CC1
2,c1ccc(cc1)-c1ccccc1-c1ccccc1,0,OASIS,c1ccc(-c2ccccc2-c2ccccc2)cc1
3,Cc1ccc(O)cc1,0,OASIS,Cc1ccc(O)cc1
4,CCOP(=S)(OCC)Oc1nc(Cl)n(n1)C(C)C,1,OASIS,CCOP(=S)(OCC)Oc1nc(Cl)n(C(C)C)n1


**Convert SMILES to RDKit molecule objects**

In [ ]:
from rdkit import Chem

master["Mol"] = master["Standard_SMILES"].apply(
    lambda x: Chem.MolFromSmiles(x) if pd.notna(x) else None
)

print("Valid molecules:", master["Mol"].notna().sum())

Valid molecules: 8622


**Calculate molecular descriptors**

In [ ]:
from rdkit.Chem import Descriptors

# Calculate molecular descriptors
master["MolecularWeight"] = master["Mol"].apply(Descriptors.MolWt)
master["LogP"] = master["Mol"].apply(Descriptors.MolLogP)
master["HDonors"] = master["Mol"].apply(Descriptors.NumHDonors)
master["HAcceptors"] = master["Mol"].apply(Descriptors.NumHAcceptors)
master["TPSA"] = master["Mol"].apply(Descriptors.TPSA)
master["RotatableBonds"] = master["Mol"].apply(Descriptors.NumRotatableBonds)

# Display the first few rows
master.head()

,SMILES,Label,Source,Standard_SMILES,Mol,MolecularWeight,LogP,HDonors,HAcceptors,TPSA,RotatableBonds
0,Cc1cc(O)c2C(=O)c3c(O)cc(O)cc3C(=O)c2c1O,1,OASIS,Cc1cc(O)c2c(c1O)C(=O)c1cc(O)cc(O)c1C2=O,<rdkit.Chem.rdchem.Mol object at 0x7be26d6a9000>,286.239,1.59282,4,6,115.06,0
1,CC1CCC(CC=1)C(C)(C)O,0,OASIS,CC1=CCC(C(C)(C)O)CC1,<rdkit.Chem.rdchem.Mol object at 0x7be26d6a91c0>,154.253,2.50370,1,1,20.23,1
2,c1ccc(cc1)-c1ccccc1-c1ccccc1,0,OASIS,c1ccc(-c2ccccc2-c2ccccc2)cc1,<rdkit.Chem.rdchem.Mol object at 0x7be26d6a9150>,230.310,5.02060,0,0,0.00,2
3,Cc1ccc(O)cc1,0,OASIS,Cc1ccc(O)cc1,<rdkit.Chem.rdchem.Mol object at 0x7be26d6a9230>,108.140,1.70062,1,1,20.23,0
4,CCOP(=S)(OCC)Oc1nc(Cl)n(n1)C(C)C,1,OASIS,CCOP(=S)(OCC)Oc1nc(Cl)n(C(C)C)n1,<rdkit.Chem.rdchem.Mol object at 0x7be26d6a92a0>,313.747,3.18870,0,6,58.40,7


**Generate Morgan fingerprints**

In [ ]:
from rdkit.Chem import AllChem
import numpy as np

# Generate Morgan fingerprints (ECFP4, 2048 bits)
fingerprints = []

for mol in master["Mol"]:
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
    fingerprints.append(list(fp))

# Convert fingerprints to a DataFrame
fp_df = pd.DataFrame(fingerprints, columns=[f"FP_{i}" for i in range(2048)])

print(fp_df.shape)
fp_df.head()

Streaming output truncated to the last 5000 lines.
[11:24:57] DEPRECATION WARNING: please use MorganGenerator
[11:24:57] DEPRECATION WARNING: please use MorganGenerator
[11:24:57] DEPRECATION WARNING: please use MorganGenerator
[11:24:57] DEPRECATION WARNING: please use MorganGenerator
[11:24:57] DEPRECATION WARNING: please use MorganGenerator
[11:24:57] DEPRECATION WARNING: please use MorganGenerator
[11:24:57] DEPRECATION WARNING: please use MorganGenerator
[11:24:57] DEPRECATION WARNING: please use MorganGenerator
[11:24:57] DEPRECATION WARNING: please use MorganGenerator
[11:24:57] DEPRECATION WARNING: please use MorganGenerator
[11:24:57] DEPRECATION WARNING: please use MorganGenerator
[11:24:57] DEPRECATION WARNING: please use MorganGenerator
[11:24:57] DEPRECATION WARNING: please use MorganGenerator
[11:24:57] DEPRECATION WARNING: please use MorganGenerator
[11:24:57] DEPRECATION WARNING: please use MorganGenerator
[11:24:57] DEPRECATION WARNING: please use MorganGenerator
[11:2

(8622, 2048)


,FP_0,FP_1,FP_2,FP_3,FP_4,FP_5,FP_6,FP_7,FP_8,FP_9,...,FP_2038,FP_2039,FP_2040,FP_2041,FP_2042,FP_2043,FP_2044,FP_2045,FP_2046,FP_2047
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


**Combine descriptors and fingerprints**

In [ ]:
# Combine descriptors and fingerprints
features = pd.concat([
    master[[
        "Label",
        "MolecularWeight",
        "LogP",
        "HDonors",
        "HAcceptors",
        "TPSA",
        "RotatableBonds"
    ]],
    fp_df
], axis=1)

print(features.shape)
features.head()

(8622, 2055)


,Label,MolecularWeight,LogP,HDonors,HAcceptors,TPSA,RotatableBonds,FP_0,FP_1,FP_2,...,FP_2038,FP_2039,FP_2040,FP_2041,FP_2042,FP_2043,FP_2044,FP_2045,FP_2046,FP_2047
0,1,286.239,1.59282,4,6,115.06,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,154.253,2.50370,1,1,20.23,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,230.310,5.02060,0,0,0.00,2,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,108.140,1.70062,1,1,20.23,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,313.747,3.18870,0,6,58.40,7,0,1,0,...,0,0,0,0,0,0,0,0,0,0


**Save the complete feature dataset**

In [ ]:
features.to_csv("master_features.csv", index=False)

print("Feature dataset saved successfully!")

Feature dataset saved successfully!


**Extract descriptor features**

In [ ]:
descriptors = master[[
    "Label",
    "MolecularWeight",
    "LogP",
    "HDonors",
    "HAcceptors",
    "TPSA",
    "RotatableBonds"
]]

descriptors.to_csv("molecular_descriptors.csv", index=False)

print("Descriptors saved.")

Descriptors saved.


**Download descriptor dataset**

In [ ]:
from google.colab import files
files.download("molecular_descriptors.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Save Morgan fingerprints**

In [ ]:
fp_df.to_csv("morgan_fingerprints.csv", index=False)

print("Fingerprints saved.")

Fingerprints saved.


**Download Morgan fingerprints**

In [ ]:
files.download("morgan_fingerprints.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Download the complete feature dataset**

In [ ]:
files.download("master_features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>